## RQ1: Predictive Performance & Key Lending Drivers
#### Group 6 | Francesco Biedermann (FB2709) | Brian Hsu (CH4004) | Phoebe Zhao (PYZ2001) 
#### APAN5205 Applied Machine Learning 2 | Prof. Andrew Assing

This notebook addresses Research Question 1: *"To what extent can mortgage approval outcomes be predicted using applicant, loan, and property characteristics available at the time of application, and which factors appear to play the largest role in these decisions?"*

We train and compare three classification models:
- Logistic Regression (with Lasso and Ridge regularization)
- Decision Tree
- Random Forest (using 5-fold cross-validation for hyperparameter tuning)

Demographic variables are excluded from the predictive models and reserved for the fairness analysis in RQ2.

### Step 1 | Setup and Data Loading
We begin by importing the required libraries and loading the cleaned HMDA dataset produced during data preparation. The dataset contains 433,173 mortgage application records with 32 columns and zero missing values.

In [1]:
# Step 1: Setup and Data Loading
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.metrics import confusion_matrix, roc_curve, ConfusionMatrixDisplay

# Load cleaned data
df = pd.read_csv('../data/hmda_2024_cleaned.csv')

print('Dataset shape:', df.shape)
print('Approval rate:', round(df['approved'].mean()*100, 2), '%')
print('\nTarget distribution:')
print(df['approved'].value_counts())

Dataset shape: (433173, 32)
Approval rate: 77.36 %

Target distribution:
approved
1    335087
0     98086
Name: count, dtype: int64


**Observations:** The dataset contains 433,173 mortgage applications with a 77.36% approval rate.

### Step 2 | Feature Selection and Preparation
We separate the target variable from the features and apply several preparation steps:

- **Demographic exclusion:** The derived race, ethnicity, and sex columns are excluded from the predictive models, consistent with our research design. These variables are reserved for the fairness analysis in RQ2.
- **Constant feature removal:** The `reverse_mortgage` column contains only a single value (2) after data cleaning and provides no discriminative information. It is dropped.
- **Leakage removal:** The `interest_rate_missing` flag is excluded. During data cleaning, we found that interest rates are only assigned after an application is approved, which means this flag is missing for nearly 100% of denied applications. Including it would allow the model to trivially predict the outcome rather than learning meaningful patterns from application-time features. The remaining missingness flags for debt-to-income ratio and combined loan-to-value ratio are retained, as their missingness patterns are not structurally determined by the outcome.
- **One-hot encoding:** The `state_code` column (54 U.S. states and territories) is one-hot encoded with `drop_first=True` to avoid multicollinearity.

In [3]:
# Step 2: Feature Selection and Preparation

# Define target
y = df['approved']

# Columns to exclude from modeling
drop_cols = [
    'approved',
    'derived_race',
    'derived_ethnicity',
    'derived_sex',
    'reverse_mortgage',
    'interest_rate_missing'
]

X = df.drop(columns=drop_cols)

print('Dropping reverse_mortgage — only one unique value:', df['reverse_mortgage'].unique())
print('Dropping interest_rate_missing — near-perfect proxy for approval status')

# One-hot encode state_code
X = pd.get_dummies(X, columns=['state_code'], drop_first=True)

print('\nFeature matrix shape:', X.shape)
print('Number of features:', X.shape[1])

Dropping reverse_mortgage — only one unique value: [2]
Dropping interest_rate_missing — near-perfect proxy for approval status

Feature matrix shape: (433173, 78)
Number of features: 78


**Observations:** After excluding demographic variables, the constant `reverse_mortgage` column and the leaky `interest_rate_missing` flag, we are left with 78 features (25 core features plus 53 state dummy variables).